# Data Exploration and Deep Preprocessing


In [21]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')


In [22]:
crm_accounts = pd.read_csv('../dataset/crm_accounts.csv')
crm_contacts = pd.read_csv('../dataset/crm_contacts.csv')
crm_deals = pd.read_csv('../dataset/crm_deals.csv')
contact_emp = pd.read_csv('../dataset/contact_employment_history.csv')

# --- Clean CRM Accounts ---
crm_accounts['paket'] = crm_accounts['paket'].fillna('Unknown')
crm_accounts['champion_contact_id'] = crm_accounts['champion_contact_id'].fillna('No Champion')
# Since NPS and Health Score have string types, we'll fill missing with 'Unknown' 
# to achieve 0 missing values without breaking the text.
crm_accounts['nps_terakhir'] = crm_accounts['nps_terakhir'].fillna('Unknown')
crm_accounts['health_score_dashboard'] = crm_accounts['health_score_dashboard'].fillna('Unknown')

# --- Clean CRM Deals ---
crm_deals['stage_sejak'] = pd.to_datetime(crm_deals['stage_sejak'], errors='coerce').fillna(pd.Timestamp('2099-12-31'))
crm_deals['dibuat'] = pd.to_datetime(crm_deals['dibuat'], errors='coerce').fillna(pd.Timestamp('2099-12-31'))
crm_deals['alasan_kalah'] = crm_deals['alasan_kalah'].fillna('Not Applicable/Unknown')
crm_deals['kompetitor'] = crm_deals['kompetitor'].fillna('No Competitor')

# --- Clean Contact Employment History ---
contact_emp['account_id'] = contact_emp['account_id'].fillna('External_Org')
contact_emp['mulai'] = pd.to_datetime(contact_emp['mulai'], errors='coerce')
# For missing 'selesai' (End Date), we fill with '2099-12-31' to indicate Present day
contact_emp['selesai'] = pd.to_datetime(contact_emp['selesai'], errors='coerce').fillna(pd.Timestamp('2099-12-31'))
contact_emp['is_current'] = contact_emp['selesai'] == pd.Timestamp('2099-12-31')


In [23]:
features = pd.read_csv('../dataset/features.csv')
feature_usage = pd.read_csv('../dataset/feature_usage_monthly.csv')
releases = pd.read_csv('../dataset/releases.csv')
bugs = pd.read_csv('../dataset/bugs.csv')
product_usage_daily = pd.read_csv('../dataset/product_usage_daily.csv', nrows=10000)

# --- Clean Features ---
features['target_awal'] = features['target_awal'].fillna('TBD')
features['target_terkini'] = features['target_terkini'].fillna('TBD')
features['catatan'] = features['catatan'].fillna('No Notes')

# --- Clean Bugs ---
bugs['dibuat'] = pd.to_datetime(bugs['dibuat'], errors='coerce')
bugs['is_resolved'] = ~bugs['selesai'].isnull()
# Calculate duration first while it's still datetime/NaT
temp_selesai = pd.to_datetime(bugs['selesai'], errors='coerce')
bugs['waktu_penyelesaian_hari'] = (temp_selesai - bugs['dibuat']).dt.days.fillna(-1) # -1 means unresolved
# Now fill the actual date with a placeholder date so missing values = 0
bugs['selesai'] = temp_selesai.fillna(pd.Timestamp('2099-12-31'))

# --- Clean product_usage_daily ---
if 'transaksi_offline_tersinkron' in product_usage_daily.columns:
    product_usage_daily['transaksi_offline_tersinkron'] = product_usage_daily['transaksi_offline_tersinkron'].fillna(0)


In [24]:
support_tickets = pd.read_csv('../dataset/support_tickets.csv')
decision_log = pd.read_csv('../dataset/decision_log.csv')

# --- Clean Support Tickets ---
support_tickets['outlet_id'] = support_tickets['outlet_id'].fillna('HQ / No Outlet')
support_tickets['pelapor_contact_id'] = support_tickets['pelapor_contact_id'].fillna('Unknown / Anonymous')
support_tickets['bug_id'] = support_tickets['bug_id'].fillna('No Bug')

support_tickets['dibuat'] = pd.to_datetime(support_tickets['dibuat'], errors='coerce')
support_tickets['is_resolved'] = ~support_tickets['diselesaikan'].isnull()

temp_diselesaikan = pd.to_datetime(support_tickets['diselesaikan'], errors='coerce')
support_tickets['resolusi_hari'] = (temp_diselesaikan - support_tickets['dibuat']).dt.days.fillna(-1) # -1 means unresolved
support_tickets['diselesaikan'] = temp_diselesaikan.fillna(pd.Timestamp('2099-12-31'))

# --- Clean Decision Log ---
decision_log['deal_id'] = decision_log['deal_id'].fillna('Non-Deal Decision')
decision_log['nilai'] = decision_log['nilai'].fillna(0)
decision_log['bukti_interaction_id'] = decision_log['bukti_interaction_id'].fillna('No Proof/Log')
decision_log['fitur_dijanjikan'] = decision_log['fitur_dijanjikan'].fillna('None')
decision_log['status_janji'] = decision_log['status_janji'].fillna('Not Applicable')


In [25]:
contracts_billing = pd.read_csv('../dataset/contracts_billing.csv')

# --- Clean Contracts & Billing ---
contracts_billing['decision_id'] = contracts_billing['decision_id'].fillna('No Decision Linked')
if 'tanggal_mulai' in contracts_billing.columns and 'tanggal_akhir' in contracts_billing.columns:
    contracts_billing['tanggal_mulai'] = pd.to_datetime(contracts_billing['tanggal_mulai'], errors='coerce').fillna(pd.Timestamp('2099-12-31'))
    contracts_billing['tanggal_akhir'] = pd.to_datetime(contracts_billing['tanggal_akhir'], errors='coerce').fillna(pd.Timestamp('2099-12-31'))


In [ ]:
# FINAL CHECK
for name, df in zip(
    ['crm_accounts', 'crm_deals', 'contact_employment_history', 'features', 'bugs', 'support_tickets', 'decision_log', 'contracts_billing', 'product_usage_daily'],
    [crm_accounts, crm_deals, contact_emp, features, bugs, support_tickets, decision_log, contracts_billing, product_usage_daily]
):
    print(f"{name}: Total missing values = {df.isnull().sum().sum()}")


crm_accounts: Total missing values = 0
crm_deals: Total missing values = 0
contact_employment_history: Total missing values = 0
features: Total missing values = 0
bugs: Total missing values = 0
support_tickets: Total missing values = 0
decision_log: Total missing values = 0
contracts_billing: Total missing values = 0
product_usage_daily: Total missing values = 0


In [27]:
import os

# Create the clean dataset folder if it doesn't exist
os.makedirs('../dataset_clean', exist_ok=True)

# Map the final cleaned dataframes to their respective file names
datasets_to_save = {
    'crm_accounts.csv': crm_accounts,
    'crm_contacts.csv': crm_contacts,
    'crm_deals.csv': crm_deals,
    'contact_employment_history.csv': contact_emp,
    'features.csv': features,
    'feature_usage_monthly.csv': feature_usage,
    'releases.csv': releases,
    'bugs.csv': bugs,
    'support_tickets.csv': support_tickets,
    'decision_log.csv': decision_log,
    'contracts_billing.csv': contracts_billing,
    'product_usage_daily.csv': product_usage_daily
}

# Save each dataframe to the folder
for filename, df in datasets_to_save.items():
    output_path = f'../dataset_clean/{filename}'
    df.to_csv(output_path, index=False)
    print(f"Saved clean data: {output_path}")

print("\nSuccess! All clean data has been exported to the dataset_clean folder.")


Saved clean data: ../dataset_clean/crm_accounts.csv
Saved clean data: ../dataset_clean/crm_contacts.csv
Saved clean data: ../dataset_clean/crm_deals.csv
Saved clean data: ../dataset_clean/contact_employment_history.csv
Saved clean data: ../dataset_clean/features.csv
Saved clean data: ../dataset_clean/feature_usage_monthly.csv
Saved clean data: ../dataset_clean/releases.csv
Saved clean data: ../dataset_clean/bugs.csv
Saved clean data: ../dataset_clean/support_tickets.csv
Saved clean data: ../dataset_clean/decision_log.csv
Saved clean data: ../dataset_clean/contracts_billing.csv
Saved clean data: ../dataset_clean/product_usage_daily.csv

Success! All clean data has been exported to the dataset_clean folder.
